# 03 · Aggregation and tract-table checks

Build steps 3 and 4: every Overture feature is reduced to a point and assigned to the census tract that contains it (`coverage_risk.aggregate.count_points`), then counts, ACS, LODES and references are joined into one row per tract (`scripts/02_build_tract_table.py`). Did anything get lost, counted twice or joined wrongly, and does a fresh run reproduce the published table?

The same rules are enforced in code: `coverage_risk/checks.py` (run by `02_build_tract_table.py` before it saves) and `tests/test_aggregate.py`, `tests/test_tract_table.py`.

In [1]:
import warnings, os, json, glob
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name in ("eda", "notebooks") else Path.cwd()
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, duckdb, pyarrow.parquet as pq
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
REG = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx", "eastern-wa"]
COL = dict(zip(REG, ["#e07b39", "#c0504d", "#6a9e4f", "#3b6fb6", "#8e6bbf"]))

def challenge_root():
    """data/raw/challenge (Windows junctions made by fetch_data.ps1), or the same files in ../MapEquityChallenge/data
    where the junction cannot be followed (e.g. from Linux/WSL)."""
    p = Path("data/raw/challenge")
    try:
        if (p / "reference" / "maricopa-az").is_dir():
            return p
    except OSError:
        pass
    return Path("../MapEquityChallenge/data")
CH = challenge_root(); RAW = Path("data/raw")
TAB, FIG = Path("eda/tables"), Path("eda/figures"); TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()
con = duckdb.connect(); con.sql("SET threads=4")
study = pd.concat([pd.read_csv(CH / f"reference/{r}/{r}-sample-submission.csv", dtype={"GEOID": str}, usecols=["GEOID"]).assign(region=r) for r in REG])
print("challenge data read from:", CH.resolve(), "| study tracts:", len(study))
P = Path("data/processed")
try:
    con.sql("LOAD spatial")
except Exception:
    con.sql("INSTALL spatial"); con.sql("LOAD spatial")

challenge data read from: /sessions/rcw-013hmcfk1naa7pch2jpe8lbk/mnt/MapEquityChallenge/data | study tracts: 9794


## 1. Are all raw features assigned to a study tract?

In [2]:
b = pd.read_parquet(P / "overture_r2026a_buildings.parquet"); p = pd.read_parquet(P / "overture_r2026a_places.parquet")
reg = pd.read_parquet(P / "tracts_all.parquet", columns=["GEOID", "region"])
b, p = b.merge(reg, on="GEOID"), p.merge(reg, on="GEOID")
rows = []
for r in REG:
    for layer, f, assigned in [("buildings", f"{r}-overture-buildings.parquet", b.loc[b.region == r, "r2026a_bldg"].sum()),
                               ("places", f"{r}-overture-pois.parquet", p.loc[p.region == r, "r2026a_poi"].sum())]:
        raw = pq.ParquetFile(CH / "reference" / r / f).metadata.num_rows
        rows.append({"region": r, "layer": layer, "raw_features": raw, "assigned": int(assigned), "unassigned": raw - int(assigned), "share_unassigned": (raw - assigned) / raw})
asg = pd.DataFrame(rows); asg.to_csv(TAB / "03_assignment_completeness.csv", index=False)
display(asg.style.format({"raw_features": "{:,.0f}", "assigned": "{:,.0f}", "unassigned": "{:,.0f}", "share_unassigned": "{:.4%}"}))
print("totals:", asg.groupby("layer")[["raw_features", "assigned", "unassigned"]].sum().to_dict("index"))

,region,layer,raw_features,assigned,unassigned,share_unassigned
0,maricopa-az,buildings,"2,908,224","2,908,224",0,0.0000%
1,maricopa-az,places,"300,046","300,044",2,0.0007%
2,northern-ca,buildings,"1,164,724","1,164,723",1,0.0001%
3,northern-ca,places,"116,897","116,893",4,0.0034%
4,eastern-ok,buildings,"2,551,694","2,551,694",0,0.0000%
5,eastern-ok,places,"207,369","207,367",2,0.0010%
6,south-central-tx,buildings,"11,463,801","11,463,800",1,0.0000%
7,south-central-tx,places,"1,300,334","1,300,281",53,0.0041%
8,eastern-wa,buildings,"1,081,346","1,081,345",1,0.0001%
9,eastern-wa,places,"76,491","76,490",1,0.0013%


totals: {'buildings': {'raw_features': 19169789, 'assigned': 19169786, 'unassigned': 3}, 'places': {'raw_features': 2001137, 'assigned': 2001075, 'unassigned': 62}}


**Takeaway.** Of 19,169,789 raw buildings, 19,169,786 are assigned to a study tract (3 unassigned); of 2,001,137 places, 2,001,075 (62 unassigned, 0.003%). The challenge extracts were already clipped to the study area, so practically nothing is lost in this step.

## 2. Where do the unassigned places go? (full spatial join, all 2 million places)

In [3]:
rows, left = [], []
for r in REG:
    import geopandas as gpd
    g = gpd.read_parquet(CH / f"strata/{r}/{r}-census-tracts.parquet").to_crs(4326)
    sub = set(study.loc[study.region == r, "GEOID"])
    con.register("tdf", pd.DataFrame({"GEOID": g.GEOID, "study": g.GEOID.isin(sub), "wkb": g.geometry.to_wkb()}))
    con.sql("CREATE OR REPLACE TABLE tr AS SELECT GEOID, study, ST_GeomFromWKB(wkb) geom FROM tdf")
    con.sql(f"CREATE OR REPLACE TABLE pts AS SELECT id, ST_Point(ST_X(geometry), ST_Y(geometry)) pt FROM read_parquet('{CH / 'reference' / r / (r + '-overture-pois.parquet')}')")
    con.sql("CREATE OR REPLACE TABLE hit AS SELECT p.id, max(t.study::int) in_study, count(*) n FROM pts p JOIN tr t ON ST_Contains(t.geom, p.pt) GROUP BY 1")
    q = con.sql("""SELECT (SELECT count(*) FROM pts) total, count(*) FILTER (WHERE in_study = 1) in_study,
                   count(*) FILTER (WHERE in_study = 0) in_excluded_tract, max(n) max_tracts_per_point FROM hit""").df().iloc[0]
    un = con.sql("""SELECT p.id, ST_X(p.pt) x, ST_Y(p.pt) y,
                    (SELECT bool_or(ST_Intersects(ST_Boundary(t.geom), p.pt)) FROM tr t) on_boundary,
                    (SELECT min(ST_Distance(t.geom, p.pt)) FROM tr t WHERE t.study) deg_to_study
                    FROM pts p WHERE p.id NOT IN (SELECT id FROM hit)""").df()
    rows.append({"region": r, **q, "outside_every_tract": len(un), "of_which_on_a_boundary": int(un.on_boundary.sum())})
    left.append(un.assign(region=r))
ua = pd.DataFrame(rows).set_index("region"); ua.to_csv(TAB / "03_unassigned_places.csv")
display(ua.style.format("{:,.0f}"))
un = pd.concat(left, ignore_index=True); un["km_to_study_area"] = un.deg_to_study * 111
print(f"places outside every tract: {len(un)}; distance to the nearest study tract: median {un.km_to_study_area.median():.2f} km, max {un.km_to_study_area.max():.1f} km")

,total,in_study,in_excluded_tract,max_tracts_per_point,outside_every_tract,of_which_on_a_boundary
region,,,,,,
maricopa-az,"300,046","300,044",0,1,2,2
northern-ca,"116,897","116,893",0,1,4,4
eastern-ok,"207,369","207,367",0,1,2,2
south-central-tx,"1,300,334","1,300,281",3,1,50,50
eastern-wa,"76,491","76,490",0,1,1,1


places outside every tract: 59; distance to the nearest study tract: median 0.00 km, max 0.0 km


**Takeaway.** A full spatial join of all 2 million places explains every unassigned one: 3 fall in the seven Texas tracts deliberately left out of the study, and the other 59 lie exactly on a boundary between tracts. The assignment uses ST_Contains, which counts a point on a shared edge in neither tract, so such features are dropped rather than double-counted; no place falls into two tracts. tests/test_aggregate.py pins this behaviour down on a hand-made example. At 59 places out of 2 million it does not affect any result, but a tie-break rule (e.g. assign boundary points to the lower GEOID) would make the count exact.

## 3. Schema and consistency checks on the rebuilt tract table

In [4]:
import sys; sys.path.insert(0, "src")
from coverage_risk.checks import check_tract_table
tt = pd.read_parquet(P / "tracts_all.parquet")
res = check_tract_table(tt, expected_geoids=study.GEOID)
chk = pd.Series(res, name="result").to_frame(); chk.to_csv(TAB / "03_tract_table_checks.csv"); display(chk)

,result
GEOID unique,ok
GEOID is 11 digits,ok
tract list matches the study list,ok
no missing structural inputs,ok
counts are non-negative,ok
"shares lie in [0, 1]",ok
buildings >= 450 sq ft <= all buildings,ok
source counts add up to buildings,ok
confident / multi-source places <= places,ok
USA Structures missing for <= 1% of tracts,ok


**Takeaway.** The rebuilt tract table passes every check: 9,794 unique 11-digit tract IDs matching the study list, no missing structural inputs, no negative counts, all shares between 0 and 1, buildings over 450 sq ft never exceeding all buildings, source counts adding up to the building count, and USA Structures present for all but one tract. scripts/02_build_tract_table.py now runs these checks before saving and stops if one fails.

## 4. Does a fresh run reproduce the published table?

In [5]:
pub = pd.read_csv("deliverables/dataset/tracts_model_table.csv", dtype={"GEOID": str, "county": str})
m = pub.merge(tt, on="GEOID", suffixes=("_pub", "_new"))
rows = []
for c in [c for c in pub.columns if c != "GEOID"]:
    a, n = m[c + "_pub"], m[c + "_new"]
    if a.dtype == object or n.dtype == object:
        same = (a.astype(str).replace({"None": "nan"}) == n.astype(str).replace({"None": "nan", "NaT": "nan"}))
        rows.append({"column": c, "identical": same.mean(), "max_abs_diff": np.nan})
    else:
        a, n = a.astype(float), n.astype(float)
        rows.append({"column": c, "identical": np.isclose(a, n, rtol=1e-9, atol=1e-9, equal_nan=True).mean(), "max_abs_diff": np.nanmax(np.abs(a - n))})
rp = pd.DataFrame(rows); rp.to_csv(TAB / "03_reproducibility.csv", index=False)
print(f"rows: published {len(pub):,}, rebuilt {len(tt):,}; columns identical in every tract: {(rp.identical == 1).sum()} of {len(rp)}")
diff = rp[rp.identical < 1]; display(diff)
if len(diff):
    cols = diff.column.tolist(); bad = m[[c + "_pub" for c in cols]].astype(str).values != m[[c + "_new" for c in cols]].astype(str).values
    display(m.loc[bad.any(axis=1), ["GEOID"] + [c + s for c in cols for s in ("_pub", "_new")]])

rows: published 9,794, rebuilt 9,794; columns identical in every tract: 111 of 111


,column,identical,max_abs_diff


**Takeaway.** A fresh run on this machine reproduces the published tract table exactly: same 9,794 rows and all 111 columns identical in every tract (the only apparent difference was one missing image date written as None in one file and NaN in the other). The data side of the project is reproducible from public sources.

## 5. Reference joins

In [6]:
ref = pd.read_parquet(P / "usa_structures_tract.parquet"); zbp = pd.read_parquet(P / "zbp_tract.parquet"); cbp = pd.read_parquet(P / "cbp_county.parquet")
cov = pd.DataFrame({"USA Structures count": tt.ref_bldg.notna(), "ZIP Business Patterns allocation": tt.ref_est_zbp.notna(),
                    "County Business Patterns": tt.county.isin(cbp.county)})
c = cov.groupby(tt.region).mean().reindex(REG); c.loc["all"] = cov.mean(); c.to_csv(TAB / "03_reference_coverage.csv"); display(c.style.format("{:.2%}"))
print(f"USA Structures: {len(ref):,} tracts in the 6 states, {ref.ref_bldg.sum():,.0f} structures; study tracts {tt.ref_bldg.sum():,.0f}")
print("study tracts without a USA Structures count:", tt.loc[tt.ref_bldg.isna(), "GEOID"].tolist())
print(f"imagery dates of USA Structures in the study: {tt.ref_img_min.min()} to {tt.ref_img_max.max()}")

,USA Structures count,ZIP Business Patterns allocation,County Business Patterns
region,,,
maricopa-az,100.00%,99.94%,100.00%
northern-ca,100.00%,100.00%,100.00%
eastern-ok,100.00%,100.00%,100.00%
south-central-tx,99.98%,99.95%,99.98%
eastern-wa,100.00%,100.00%,100.00%
all,99.99%,99.96%,99.99%


USA Structures: 21,356 tracts in the 6 states, 31,228,307 structures; study tracts 16,177,725
study tracts without a USA Structures count: ['48057990000']
imagery dates of USA Structures in the study: 2006-01-01 00:00:00 to 2021-05-16 00:00:00


**Takeaway.** References join almost completely: USA Structures for 9,793 of 9,794 tracts (the missing one, 48057990000, is a zero-land water tract), ZIP Business Patterns for all but 4 tracts, County Business Patterns for all but one. One substantive point for interpretation: USA Structures imagery dates from 2006 to 2021, years older than the 2026 Overture map, so buildings built since then are in Overture but not in the reference. This is one reason Overture usually exceeds the reference (median ratio 1.05) and it makes the reference conservative for detecting gaps.

## 6. Pipeline status: which outputs exist?

In [7]:
need = {"overture_r2026a_buildings.parquet": "01 --release r2026a", "overture_r2026a_places.parquet": "01 --release r2026a",
        "overture_r2026b_buildings.parquet": "01 --release r2026b", "overture_r2026b_places.parquet": "01 --release r2026b",
        "overture_r2023_buildings.parquet": "01 --release r2023", "usa_structures_tract.parquet": "00b", "zbp_tract.parquet": "00c",
        "cbp_county.parquet": "00c", "tracts_all.parquet": "02", "strata_audit.parquet": "02"}
stt = pd.DataFrame([{"file": f, "made_by": s, "present": (P / f).exists()} for f, s in need.items()])
ck = sorted(x.name for x in P.glob("_ck_*.parquet"))
display(stt); print("checkpoints (partial runs resume from these):", [c.replace("_ck_", "").replace(".parquet", "") for c in ck])

,file,made_by,present
0,overture_r2026a_buildings.parquet,01 --release r2026a,True
1,overture_r2026a_places.parquet,01 --release r2026a,True
2,overture_r2026b_buildings.parquet,01 --release r2026b,False
3,overture_r2026b_places.parquet,01 --release r2026b,False
4,overture_r2023_buildings.parquet,01 --release r2023,False
5,usa_structures_tract.parquet,00b,True
6,zbp_tract.parquet,00c,True
7,cbp_county.parquet,00c,True
8,tracts_all.parquet,02,True
9,strata_audit.parquet,02,True


checkpoints (partial runs resume from these): ['r2026a_buildings_eastern-ok', 'r2026a_buildings_eastern-wa', 'r2026a_buildings_maricopa-az', 'r2026a_buildings_northern-ca', 'r2026a_buildings_south-central-tx', 'r2026a_places_eastern-ok', 'r2026a_places_eastern-wa', 'r2026a_places_maricopa-az', 'r2026a_places_northern-ca', 'r2026a_places_south-central-tx', 'r2026b_buildings_eastern-ok', 'r2026b_buildings_eastern-wa', 'r2026b_buildings_maricopa-az', 'r2026b_buildings_northern-ca']


**Takeaway.** Everything needed for the 2026-08 analysis exists. The 2026-10 counts stopped after four regions (south-central Texas is missing) and the 2023 counts have not run; rerun scripts/fetch_data.ps1 -Stream, which resumes from the checkpoints listed above. Overture deletes old releases after about 60 days, so this should be done before late December 2026.

## Summary

Steps 3 and 4 are sound: almost every feature is assigned, the few that are not are explained (boundary points and excluded tracts), the tract table passes every consistency check, and a fresh run reproduces the published table exactly. The only remaining data task is to finish the 2026-10 and 2023 counts.